In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
### 所需要的所有文件：
# 物流的发货清单
# 财务的发货清单
# 核算价
# PLM的生命周期全表
### 输出的所有文件
# 统计周期内产品核算价汇总 用于物料精简报告，因为里面有国内国外
# 合并物流-财务-产品组-核算价-国内-用于低效-长尾  ，用于低效-长尾报告，里面只有国内
# 单型号贡献-统计值



### MAP关系汇总1、渠道对照 2、最终表格产品类别对应的产品组集合 3、产品组集合

In [4]:
# 物流的渠道对照关系清洗用
month = 202512
Channel_map = {
'工程': '工程',
'零售':'零售',
'电商不可售':'电商',
'电商':'电商',
'内部处理通用':'电商',
'借出渠道':'非零售工程电商',
'新品':'电商',
'战略电商':'电商',
'转出渠道':'非零售工程电商',
'每誉':'每誉',
'渠道':'非零售工程电商',
'海外':'海外',
'调出渠道':'非零售工程电商',
'非零售工程电商':'非零售工程电商',
}
#用于合并计算
productgroupset_map = {
    '吸油烟机':['吸油烟机'],
    '灶具':['灶具'],
    '烤箱':['烤箱'],
    '蒸箱':['蒸箱'],
    '微波炉':['微波炉'],
    '蒸烤烹饪机':['蒸烤烹饪机'],
    '蒸烤微烹饪机':['蒸烤微烹饪机'],
    '蒸微':['蒸微'],
    '蒸烤微合计':['烤箱','蒸箱','微波炉','蒸烤烹饪机','蒸烤微烹饪机','蒸微'],
    '灶消烹饪机':['灶消烹饪机'],
    '灶蒸烹饪机':['灶蒸烹饪机'],
    '灶蒸烤烹饪机':['灶蒸烤烹饪机'],
    '灶烤烹饪机':['灶烤烹饪机'],
    '灶集成':['灶消烹饪机','灶蒸烹饪机','灶蒸烤烹饪机','灶烤烹饪机'],
    '烹饪产品线合计':['灶具','烤箱','蒸箱','微波炉','蒸烤烹饪机','蒸烤微烹饪机','蒸微','灶消烹饪机','灶蒸烹饪机','灶蒸烤烹饪机','灶烤烹饪机'],
    '消毒柜':['消毒柜'],
    '热水器':['热水器'],
    '两用炉':['两用炉'],
    '热水器两用炉合计':['热水器','两用炉'],
    '家用净水机':['家用净水机'],
    '商用净水机':['商用净水机'],
    '净热产品线合计':['热水器','两用炉','家用净水机','商用净水机'],
    '水槽洗碗机':['水槽洗碗机'],
    '嵌入式洗碗机':['嵌入式洗碗机'],
    '洗碗机产品线合计':['水槽洗碗机','嵌入式洗碗机'],
    '国内合计':['吸油烟机','灶具','烤箱','蒸箱','微波炉','蒸烤烹饪机','蒸烤微烹饪机','蒸微','灶消烹饪机','灶蒸烹饪机','灶蒸烤烹饪机','灶烤烹饪机','消毒柜','热水器','两用炉','家用净水机','商用净水机','水槽洗碗机','嵌入式洗碗机'],
}
# 统计的产品组
productgroup_list = ['吸油烟机','灶具','烤箱','蒸箱','微波炉','蒸烤烹饪机','蒸烤微烹饪机','蒸微','灶消烹饪机','灶蒸烹饪机','灶蒸烤烹饪机','灶烤烹饪机','消毒柜','热水器','两用炉','热水器两用炉合计','家用净水机','商用净水机','净热产品线合计','水槽洗碗机','嵌入式洗碗机','洗碗机产品线合计','国内合计']


### 获取物流体系的文件夹中的文件路劲和对应规则的sheetname。并将这些数据进行上写拼接（只保留了编码，渠道，发货数量）

In [5]:
#将这些excel的表格数据进行上下拼接，只要字段：商品编码、商品名称、渠道、实际出库数量
folder_path = fr"D:\000物料报表\{month}\单型号贡献-低效-长尾\物流发货"  # 请替换为实际的文件夹路径
excel_files = []
for root, dirs, files in os.walk(folder_path):
    for file in files:
        sheet_names = file[5:].replace(file[-5:],'')
        if (file.endswith('.xlsx') or file.endswith('.xls')) and not file.startswith('~$'):
            file_path = os.path.join(root, file)
            excel_files.append((file, sheet_names, file_path))
print(excel_files)

[('2025年10月明细.xlsx', '10月明细', 'D:\\000物料报表\\202512\\单型号贡献-低效-长尾\\物流发货\\2025年10月明细.xlsx'), ('2025年11月明细.xlsx', '11月明细', 'D:\\000物料报表\\202512\\单型号贡献-低效-长尾\\物流发货\\2025年11月明细.xlsx'), ('2025年12月明细.xlsx', '12月明细', 'D:\\000物料报表\\202512\\单型号贡献-低效-长尾\\物流发货\\2025年12月明细.xlsx'), ('2025年1月明细.xlsx', '1月明细', 'D:\\000物料报表\\202512\\单型号贡献-低效-长尾\\物流发货\\2025年1月明细.xlsx'), ('2025年2月明细.xlsx', '2月明细', 'D:\\000物料报表\\202512\\单型号贡献-低效-长尾\\物流发货\\2025年2月明细.xlsx'), ('2025年3月明细.xlsx', '3月明细', 'D:\\000物料报表\\202512\\单型号贡献-低效-长尾\\物流发货\\2025年3月明细.xlsx'), ('2025年4月明细.xlsx', '4月明细', 'D:\\000物料报表\\202512\\单型号贡献-低效-长尾\\物流发货\\2025年4月明细.xlsx'), ('2025年5月明细.xlsx', '5月明细', 'D:\\000物料报表\\202512\\单型号贡献-低效-长尾\\物流发货\\2025年5月明细.xlsx'), ('2025年6月明细.xlsx', '6月明细', 'D:\\000物料报表\\202512\\单型号贡献-低效-长尾\\物流发货\\2025年6月明细.xlsx'), ('2025年7月明细.xlsx', '7月明细', 'D:\\000物料报表\\202512\\单型号贡献-低效-长尾\\物流发货\\2025年7月明细.xlsx'), ('2025年8月明细.xlsx', '8月明细', 'D:\\000物料报表\\202512\\单型号贡献-低效-长尾\\物流发货\\2025年8月明细.xlsx'), ('2025年9月明细.xlsx', '9月明细', 'D:\\000物料报表\\202

In [6]:
#如果有报错请提示报错信息
df = pd.DataFrame()
for file in excel_files:
    try:
        df_temp = pd.read_excel(file[2], sheet_name=file[1])
        df_temp['发货月份'] = file[0][:8]
        print(f'成功读取{file[0]}的{file[1]}表')
    except:
        print(f'读取{file[0]}的{file[1]}表失败')
    if '实际总数量' in df_temp.columns:
        df_temp = df_temp.rename(columns={'实际总数量':'实际出库数量'})
    df_temp = df_temp[['商品编码', '渠道', '实际出库数量','发货月份']]
    df = pd.concat([df, df_temp], axis=0).reset_index(drop=True)
df = df.dropna(how='all').reset_index(drop=True)  # 仅当一行所有值都是NaN时才删除
df['商品编码'] = df['商品编码'].astype(str)
df['渠道'].value_counts()


成功读取2025年10月明细.xlsx的10月明细表
成功读取2025年11月明细.xlsx的11月明细表
成功读取2025年12月明细.xlsx的12月明细表
成功读取2025年1月明细.xlsx的1月明细表
成功读取2025年2月明细.xlsx的2月明细表
成功读取2025年3月明细.xlsx的3月明细表
成功读取2025年4月明细.xlsx的4月明细表
成功读取2025年5月明细.xlsx的5月明细表
成功读取2025年6月明细.xlsx的6月明细表
成功读取2025年7月明细.xlsx的7月明细表
成功读取2025年8月明细.xlsx的8月明细表
成功读取2025年9月明细.xlsx的9月明细表


渠道
零售        489375
工程         21166
电商         10507
电商不可售       6616
海外          2359
每誉           188
战略电商         129
新品            67
内部处理通用        15
商净            14
借出渠道          14
渠道             4
调出渠道           2
转出渠道           2
米博新零售          1
Name: count, dtype: int64

### 处理历史残留（财务的发货数据，将其转化为编码，名称，渠道，出库数量的形式）

In [5]:
# #将这些excel的表格数据进行上下拼接，只要字段：商品编码、商品名称、渠道、实际出库数量
# folder_path = fr"D:\000物料报表\{month}\单型号贡献-低效-长尾\财务发货"  # 请替换为实际的文件夹路径
# excel_files = []
# for root, dirs, files in os.walk(folder_path):
#     for file in files:
#         sheet_names = file[5:].replace(file[-5:],'')
#         if (file.endswith('.xlsx') or file.endswith('.xls')) and not file.startswith('~$'):
#             file_path = os.path.join(root, file)
#             excel_files.append((file, sheet_names, file_path))
# print(excel_files)

In [6]:
# caiwu_shouru = {'商品编码':[],'渠道':[],'实际出库数量':[]}
# for file in excel_files:
#     try:
#         df_temp = pd.read_excel(file[2], sheet_name=file[1])
#         print(f'成功读取{file[0]}的{file[1]}表')
#     except:
#         print(file[0], file[1])
#     for index,row in df_temp.iterrows():
#         if row['零售'] > 0:
#             caiwu_shouru['商品编码'].append(row['物料编码'])
#             caiwu_shouru['渠道'].append('零售')
#             caiwu_shouru['实际出库数量'].append(row['零售'])
#         if row['工程'] >0:
#             caiwu_shouru['商品编码'].append(row['物料编码'])
#             caiwu_shouru['渠道'].append('工程')
#             caiwu_shouru['实际出库数量'].append(row['工程'])
#         if row['电商'] >0:
#             caiwu_shouru['商品编码'].append(row['物料编码'])
#             caiwu_shouru['渠道'].append('电商')
#             caiwu_shouru['实际出库数量'].append(row['电商'])
#         if row['合计-发货'] - row['零售'] - row['工程'] - row['电商'] != 0:
#             caiwu_shouru['商品编码'].append(row['物料编码'])
#             caiwu_shouru['渠道'].append('非零售工程电商')
#             caiwu_shouru['实际出库数量'].append(row['合计-发货'] - row['零售'] - row['工程'] - row['电商'])

# df_caiwu = pd.DataFrame(caiwu_shouru)
# df_caiwu['商品编码'] = df_caiwu['商品编码'].astype(str)
# df_caiwu['渠道'].value_counts()


### 将财务和物流的数据进行拼接,并进行渠道清洗

In [7]:
# df0 = pd.concat([df, df_caiwu], axis=0).reset_index(drop=True)
df0 = df  # 这里只看物流
# df0.to_excel(r'C:\Users\zhangbon\Desktop\总发货.xlsx', index=False)
df0['物料编码'] = df0['商品编码'].astype(str).str[:13]
df0['渠道'] = df0['渠道'].map(Channel_map).fillna('非零售工程电商')
df0 = df0[(df0['渠道']!='无') & 
        (df0['物料编码'].str.len()>=10)
        ].reset_index(drop=True)
df0['渠道'].value_counts()
# df0


渠道
零售         489375
工程          21166
电商          17334
海外           2359
每誉            188
非零售工程电商        15
Name: count, dtype: int64

In [8]:
# df0.to_excel(r'C:\Users\zhangbon\Desktop\发货数据.xlsx', index=False)

### 将产品组信息、核算价匹配进去

In [8]:
# 复制df
df1 = df0.copy()
df1['物料编码'] = df1['物料编码'].astype(str)
df1['实际出库数量'] = df1['实际出库数量'].astype(float)

# PLM产品数据导入
df_plm = pd.read_excel(fr"D:\000物料报表\{month}\单型号贡献-低效-长尾\产品生命周期状态全表.xlsx")
df_plm[['物料号','标准型号','国内/海外']] = df_plm[['物料号','标准型号','国内/海外']].astype(str)
df_plm = df_plm[df_plm['物料号'].str.len()>10]
df_plm['物料号'] = df_plm['物料号'].apply(lambda x: x[:13])
# df_product_group


e:\python\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [9]:
#产品组对照关系
product_group_map = dict(zip(df_plm['物料号'],df_plm['产品组']))
#标准型号对照关系
product_standard = dict(zip(df_plm['物料号'],df_plm['标准型号']))
#记录国内/海外状态
product_country = dict(zip(df_plm['物料号'],df_plm['国内/海外']))
#记录生命周期状态
product_status = dict(zip(df_plm['物料号'],df_plm['产品状态']))


In [10]:
# 导入财务的核算价
df_price = pd.read_excel(fr"D:\000物料报表\{month}\单型号贡献-低效-长尾\核算价.xlsx")
df_price['产品编码'] = df_price['产品编码'].astype(str)
df_price['系统核算价'] = df_price['系统核算价'].astype(float)
# df_price
product_price_map = dict(zip(df_price['产品编码'],df_price['系统核算价']))

In [11]:
df1['产品组'] = df1['商品编码'].map(product_group_map)
df1['系统核算价'] = df1['商品编码'].map(product_price_map)
df1['核算价'] = df1['系统核算价'] * df1['实际出库数量']
df1['标准型号'] = df1['商品编码'].map(product_standard)
df1['国内/海外'] = df1['商品编码'].map(product_country)
df1['生命周期状态'] = df1['商品编码'].map(product_status)
df1 = df1[df1['商品编码'].str.startswith('10')].reset_index(drop=True)
#这里要提示哪些数据的系统核算价是空的
print(f'国内没有系统核算价的是这些数据\n{df1[(df1['系统核算价'].isnull()&(df1['国内/海外']=='国内'))]['商品编码'].drop_duplicates()}')
print(len(df1))
df1['国内/海外'].value_counts()
df1.columns

国内没有系统核算价的是这些数据
Series([], Name: 商品编码, dtype: object)
524227


Index(['商品编码', '渠道', '实际出库数量', '发货月份', '物料编码', '产品组', '系统核算价', '核算价', '标准型号',
       '国内/海外', '生命周期状态'],
      dtype='object')

### 这里导出的数据，只有整机有产品组的值，其他产品组为空,这个用于物料精简（零件分析）报告的分析，用于计算收入

In [13]:
# 物流和财务的发货收入中是包含了1012开头的样机的，PLM中是只有成品的，所以这里样机的产品组是空的，样机不在我们报告的统计范围内
# 这份导出的数据中是包含了国内国外的整机以及样机等数据，（样机等不会有产品组信息，因为PLM产品生命周期表中是没有样机的）
df1.to_excel(fr'D:\000物料报表\{month}\单物料产值\统计周期内产品核算价汇总.xlsx',index=False)


### 保留范围内的产品组(这样是会只剩下整机)、国内产品、停止发货以前（不含停止发货）、三大渠道，这里要注意补充核算价！！！！！（在这里会输出一个文件用于后续低效和长尾的分析）

In [14]:
out_df = df1.copy()
out_df = out_df[(out_df['国内/海外']=='国内') & 
                # (out_df['生命周期状态'].isin(['开发','停止生产','量产','停止销售','小批量','样机','退市预警','创建',])) &
                (out_df['渠道'].isin(['零售','工程','电商']))&
                (out_df['产品组'].isin(productgroup_list))
                ].reset_index(drop=True)
#这里要提示哪些数据的系统核算价是空的
print(f'没有系统核算价的是这些数据\n{out_df[out_df['系统核算价'].isnull()]['商品编码'].drop_duplicates()}')
print(len(out_df))
# 这里导出数据用于低效和长尾的分析。因为低效和长尾只看国内的成品情况（不包含样机，且在我们的统计产品组范围内）
out_df.to_excel(fr'D:\000物料报表\{month}\单型号贡献-低效-长尾\合并物流-财务-产品组-核算价-国内-用于低效-长尾-包含停止发货.xlsx',index=False)
out_df


没有系统核算价的是这些数据
Series([], Name: 商品编码, dtype: object)
498632


,商品编码,渠道,实际出库数量,发货月份,物料编码,产品组,系统核算价,核算价,标准型号,国内/海外,生命周期状态
0,1001001500116,零售,12.0,2025年10月,1001001500116,吸油烟机,3358.0,40296.0,Z8T,国内,停止销售
1,1009000600033,零售,1.0,2025年10月,1009000600033,蒸烤烹饪机,3450.0,3450.0,ZK50-02-F1,国内,量产
2,1009000500035,零售,3.0,2025年10月,1009000500035,灶蒸烤烹饪机,5280.0,15840.0,JZT-ZK46-X2,国内,量产
3,1001001500131,零售,6.0,2025年10月,1001001500131,吸油烟机,2988.0,17928.0,02-Z6TA,国内,停止销售
4,1002003700049,零售,5.0,2025年10月,1002003700049,灶具,2550.0,12750.0,H8B,国内,量产
...,...,...,...,...,...,...,...,...,...,...,...
498627,1018001100016,零售,-12.0,2025年9月明,1018001100016,嵌入式洗碗机,3750.0,-45000.0,JBCD7E-04-Z5,国内,量产
498628,1018001100017,电商,6.0,2025年9月明,1018001100017,嵌入式洗碗机,4500.0,27000.0,JBCD7E-04-Y1,国内,量产
498629,1018001100017,零售,-6.0,2025年9月明,1018001100017,嵌入式洗碗机,4500.0,-27000.0,JBCD7E-04-Y1,国内,量产
498630,1018001100018,电商,6.0,2025年9月明,1018001100018,嵌入式洗碗机,4500.0,27000.0,JBCD7E-04-M-Y1,国内,量产


In [15]:
# df_result1 = out_df.groupby(['产品组','渠道'],as_index=False).agg(
#                                                                 标准型号数=('标准型号','nunique'),
#                                                                 核算价=('核算价','sum'),                                                          
# )
# df_result1


### 构建最终的统计结果表

In [16]:
df_calu = pd.DataFrame()
df_calu['产品类别'] = productgroupset_map.keys()
for k,v in productgroupset_map.items():
    # 零售渠道的统计
    vals = out_df[(out_df['产品组'].isin(v))&(out_df['渠道'] == '零售')]['标准型号'].nunique()
    df_calu.loc[df_calu['产品类别']==k,'零售渠道标准型号数'] = vals
    vals = out_df[(out_df['产品组'].isin(v))&(out_df['渠道'] == '零售')]['核算价'].sum()
    df_calu.loc[df_calu['产品类别']==k,'零售渠道核算价'] = vals

    # 工程渠道的统计
    vals = out_df[(out_df['产品组'].isin(v))&(out_df['渠道'] == '工程')]['标准型号'].nunique()
    df_calu.loc[df_calu['产品类别']==k,'工程渠道标准型号数'] = vals
    vals = out_df[(out_df['产品组'].isin(v))&(out_df['渠道'] == '工程')]['核算价'].sum()
    df_calu.loc[df_calu['产品类别']==k,'工程渠道核算价'] = vals

    # 电商渠道的统计
    vals = out_df[(out_df['产品组'].isin(v))&(out_df['渠道'] == '电商')]['标准型号'].nunique()
    df_calu.loc[df_calu['产品类别']==k,'电商渠道标准型号数'] = vals
    vals = out_df[(out_df['产品组'].isin(v))&(out_df['渠道'] == '电商')]['核算价'].sum()
    df_calu.loc[df_calu['产品类别']==k,'电商渠道核算价'] = vals

    # 合计
    vals = out_df[(out_df['产品组'].isin(v))]['标准型号'].nunique()
    df_calu.loc[df_calu['产品类别']==k,'汇总标准型号数'] = vals
    vals = out_df[(out_df['产品组'].isin(v))]['核算价'].sum()
    df_calu.loc[df_calu['产品类别']==k,'汇总核算价'] = vals
df_calu

,产品类别,零售渠道标准型号数,零售渠道核算价,工程渠道标准型号数,工程渠道核算价,电商渠道标准型号数,电商渠道核算价,汇总标准型号数,汇总核算价
0,吸油烟机,95.0,4.085279e+09,70.0,7.889209e+08,112.0,2.140579e+09,165.0,7.014779e+09
1,灶具,66.0,1.841277e+09,62.0,3.951474e+08,78.0,1.327099e+09,105.0,3.563523e+09
2,烤箱,8.0,5.024228e+07,5.0,4.220300e+06,9.0,9.059010e+06,11.0,6.352159e+07
3,蒸箱,7.0,6.011374e+07,4.0,2.158600e+06,9.0,1.254247e+07,11.0,7.481481e+07
4,微波炉,4.0,1.116727e+07,3.0,1.696160e+06,2.0,3.875450e+06,5.0,1.673888e+07
5,蒸烤烹饪机,26.0,6.161324e+08,20.0,6.428179e+07,41.0,1.376373e+08,46.0,8.180516e+08
6,蒸烤微烹饪机,5.0,2.255744e+07,5.0,1.596250e+06,8.0,7.202006e+07,8.0,9.617375e+07
7,蒸微,0.0,0.000000e+00,0.0,0.000000e+00,0.0,0.000000e+00,0.0,0.000000e+00
8,蒸烤微合计,50.0,7.602132e+08,37.0,7.395310e+07,69.0,2.351343e+08,81.0,1.069301e+09
9,灶消烹饪机,3.0,3.220889e+07,1.0,7.540000e+03,3.0,5.954600e+05,4.0,3.281189e+07


#### 输出结果

In [17]:

df_calu[['零售渠道核算价','电商渠道核算价','工程渠道核算价','汇总核算价']] = df_calu[['零售渠道核算价','电商渠道核算价','工程渠道核算价','汇总核算价']]/10000
df_calu['零售单型号贡献值'] = df_calu['零售渠道核算价']/df_calu['零售渠道标准型号数']
df_calu['工程单型号贡献值'] = df_calu['工程渠道核算价']/df_calu['工程渠道标准型号数']
df_calu['电商单型号贡献值'] = df_calu['电商渠道核算价']/df_calu['电商渠道标准型号数']
df_calu['汇总单型号贡献值'] = df_calu['汇总核算价']/df_calu['汇总标准型号数']
df_calu.to_excel(fr'D:\000物料报表\{month}\单型号贡献-低效-长尾\单型号贡献-统计值-包含停止发货.xlsx',index=False)
df_calu


,产品类别,零售渠道标准型号数,零售渠道核算价,工程渠道标准型号数,工程渠道核算价,电商渠道标准型号数,电商渠道核算价,汇总标准型号数,汇总核算价,零售单型号贡献值,工程单型号贡献值,电商单型号贡献值,汇总单型号贡献值
0,吸油烟机,95.0,408527.9446,70.0,78892.0886,112.0,214057.8596,165.0,7.014779e+05,4300.294154,1127.029837,1911.230889,4251.381168
1,灶具,66.0,184127.6548,62.0,39514.7376,78.0,132709.9146,105.0,3.563523e+05,2789.812952,637.334477,1701.409162,3393.831495
2,烤箱,8.0,5024.2280,5.0,422.0300,9.0,905.9010,11.0,6.352159e+03,628.028500,84.406000,100.655667,577.469000
3,蒸箱,7.0,6011.3740,4.0,215.8600,9.0,1254.2470,11.0,7.481481e+03,858.767714,53.965000,139.360778,680.134636
4,微波炉,4.0,1116.7270,3.0,169.6160,2.0,387.5450,5.0,1.673888e+03,279.181750,56.538667,193.772500,334.777600
5,蒸烤烹饪机,26.0,61613.2433,20.0,6428.1791,41.0,13763.7335,46.0,8.180516e+04,2369.740127,321.408955,335.700817,1778.372954
6,蒸烤微烹饪机,5.0,2255.7440,5.0,159.6250,8.0,7202.0060,8.0,9.617375e+03,451.148800,31.925000,900.250750,1202.171875
7,蒸微,0.0,0.0000,0.0,0.0000,0.0,0.0000,0.0,0.000000e+00,NaN,NaN,NaN,NaN
8,蒸烤微合计,50.0,76021.3163,37.0,7395.3101,69.0,23513.4325,81.0,1.069301e+05,1520.426326,199.873246,340.774384,1320.124184
9,灶消烹饪机,3.0,3220.8890,1.0,0.7540,3.0,59.5460,4.0,3.281189e+03,1073.629667,0.754000,19.848667,820.297250
